In [ ]:
import os
os.chdir('../')  # coconut root dir

In [ ]:
from coconut.configs import TRMLoRA, TRMDelora, TRMSvft
from coconut.load_model import resume_model, load_new_model, coconut_to_adapter_config_converter, load_adapter
import torch
from pathlib import Path

device = 'cuda' if torch.cuda.is_available() else 'cpu'
dtype = torch.bfloat16 if torch.cuda.is_available() else torch.float32

# load_model_path='./outputs/trm-qwen3-0.6b_20251021-131708/checkpoint_15/pytorch_model.safetensors'
# load_model_path='./outputs/trm-qwen3-0.6b_20251021-131708/checkpoint_8/pytorch_model.safetensors'
# load_model_path ='./outputs/trm-qwen3-0.6b_20251022-065910/checkpoint_24/pytorch_model.safetensors'
save_path = Path('outputs/trmlora-qwen3-0.6b_20251024-161939/checkpoint_4/trmlora')
save_path = Path("outputs/trmsvft-qwen3-0.6b_20251029-091437/checkpoint_5/")

# load toml
f = Path(save_path) / 'coconut_config.toml'

import tomli

with open(f, 'rb') as fp:
    conf_dict = tomli.load(fp)


# could load from coconut_config.toml
conf = TRMSvft(
    **conf_dict,
    )

model_id = conf_dict['model_id']
conf

TRMSvft(project='coconut', save_path='outputs/', name='trmsvft-qwen3-0.6b', model_id='suayptalha/Qwen3-0.6B-Math-Expert', only_eval=False, load_model_path='', resume_epochs=2, use_position_ids=True, bf16=True, bf16_weight=False, opt_8b=False, load_in_4bit=False, load_in_8bit=False, cot_epochs=0, epochs_per_stage=8, max_latent_stage=3, num_epochs=6, batch_size_training=12, gradient_accumulation_steps=3, lr=0.001, weight_decay=0.03, grad_clip=10.0, scheduler='linear', debug=False, seed=42, reset_optimizer=False, loss_seq_vcr=False, collect_hs=False, max_size=5000, c_thought=1, pad_latent_to_max=True, uniform_prob=0.0, train_path='data/gsm_train.json', val_path='data/gsm_valid.json', system_prompt='', latent_token_id=None, bot_token_id=None, eot_token_id=None, eos_token_id=None, skip_stage_zero=True, eval_first_epoch=False, loss_nll_ratio_margin=False, trm_h_cycles=3, trm_l_cycles=6, trm_l_layers=2, trm_num_heads=4, trm_expansion=2.0, trm_persistent_steering=True, layers_spacing_adapter=5

In [ ]:


# peft_config = coconut_to_adapter_config_converter(conf)

tokenizer, loaded_model = load_adapter(
    model_id=model_id,
    save_dir=save_path,
    Config=type(conf),
    adapter_name="default",
)

ValueError: No `target_modules` passed but also no `target_parameters` found. Please check the values for these arguments.

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained(model_id)

## Dataset

In [ ]:
# ds
from coconut.dataset import (
    CoconutCollator,
    get_cot_latent_dataset,
    get_dataset,
    get_question_only_latent_dataset,
)
max_size = 32

base_dataset_valid = get_dataset(
    conf.val_path,
    tokenizer,
    max_size=max_size // 30 + 3,
    drop_unused=False,
    system_prompt=conf.system_prompt,
)

from scripts.run import run_ratio_eval
from coconut.eval import evaluate, get_answer_perplexity, get_answer_preference

stage=1
model.to(device=device, dtype=dtype)
run_ratio_eval(
    model,
    tokenizer,
    base_dataset_valid,
    conf,
    stage,
)


num_proc must be <= 4. Reducing num_proc to 4 for dataset of size 4.


tokenize_sample: data/gsm_valid.json (num_proc=4):   0%|          | 0/4 [00:00<?, ? examples/s]

2025-10-29 16:59:08.598 | DEBUG    | coconut.dataset:get_dataset:86 - Example row:
2025-10-29 16:59:08.599 | DEBUG    | coconut.dataset:get_dataset:87 - question_tokenized
2025-10-29 16:59:08.600 | DEBUG    | coconut.dataset:get_dataset:89 - steps_tokenized[0] <<4-2=2>>
2025-10-29 16:59:08.600 | DEBUG    | coconut.dataset:get_dataset:89 - steps_tokenized[1] <<2/.5=4>>
2025-10-29 16:59:08.600 | DEBUG    | coconut.dataset:get_dataset:89 - steps_tokenized[2] <<12/4=3>>
2025-10-29 16:59:08.601 | DEBUG    | coconut.dataset:get_dataset:89 - steps_tokenized[3] <<100*3=300>>
2025-10-29 16:59:08.601 | DEBUG    | coconut.dataset:get_dataset:90 - answer_tokenized ### 300
<|im_end|>



ImportError: cannot import name 'run_ratio_eval' from 'scripts.run' (/media/wassname/SGIronWolf/projects5/2025/fbai_coconut/scripts/run.py)

In [ ]:
latent_id = tokenizer.convert_tokens_to_ids("<|latent|>")
bot_id = tokenizer.convert_tokens_to_ids("<|start-latent|>")
eot_id = tokenizer.convert_tokens_to_ids("<|end-latent|>")
collator = CoconutCollator(tokenizer, latent_id=latent_id, label_pad_token_id=-100)
max_new_tokens = 64

dataset_gen_val = get_question_only_latent_dataset(
    stage,
    base_dataset_valid,
    conf,
    bot_id,
    latent_id,
    eot_id,
    # drop_unused=False,
)
valid_gen_dataloader = torch.utils.data.DataLoader(
    dataset_gen_val,
    num_workers=6,
    pin_memory=True,
    batch_size=conf.batch_size_training,
    collate_fn=collator,
)
r = evaluate(
    valid_gen_dataloader,
    model,
    tokenizer,
    base_dataset_valid,
    max_new_tokens=max_new_tokens,
    name=f"eval_{load_model_path}",
    dtype=dtype,
    device=device,
)

In [ ]:
from coconut.gen import gen_sample, gen

In [ ]:

# def gen(s, **kwargs):
#     if isinstance(s, str):
#         inputs = tokenizer.apply_chat_template(
#             [{'role': 'user', 'content': s}],    return_tensors='pt',
#             truncation=True, padding=True, max_length=128, return_dict=True, **kwargs
#         ).to(device)
#     elif isinstance(s, list):
#         inputs = tokenizer.apply_chat_template(
#             s,    return_tensors='pt',
#             truncation=True, padding=True, max_length=128, return_dict=True, **kwargs
#         ).to(device)
#     else:
#         raise ValueError('s should be str or list')

#     with torch.autocast(device_type='cuda', dtype=dtype):
#         inputs = {k: v.to(device=device) for k, v in inputs.items()}
#         out = model.generate(
#             input_ids=inputs["input_ids"],
#             attention_mask=inputs["attention_mask"],
#             # input_embedings=inputs["input_embeddings"],
#             max_new_tokens=64,
#             min_new_tokens=16,
#             # do_sample=True,
#             # top_p=0.9,
#             # temperature=0.7,
#             do_sample=False,
#         )

#     s = tokenizer.batch_decode(out, skip_special_tokens=False)[0]
#     print('---input---')
#     print(s)
#     print('---output---')
#     return s

# # s='Tell me a long story about where is coconut?'
# # think_suffix = '<|start-latent|><|latent|><|end-latent|>'
# # (gen(s+think_suffix))
# # (gen(s));

In [ ]:
# try differen't lengthso f latent
for l in range(0, 10, 2):
    latent_tokens = '<|start-latent|>' + '<|latent|>' * l + '<|end-latent|>'
    s=[
       {'role':'user', 'content':'What is two plus two but wrong and french?'+latent_tokens},]
    print(f'--- Generating with {l} latent tokens ---')
    gen(s, add_generation_prompt=True)

In [ ]:
# try differen't lengthso f latent
for l in range(0, 10, 2):
    latent_tokens = '<|start-latent|>' + '<|latent|>' * l + '<|end-latent|>'
    s=[
       {'role':'user', 'content':'What is two plus two but wrong and french?'},
       {'role':'assistant', 'content':'Sure thing meatbag'+latent_tokens}]
    print(f'--- Generating with {l} latent tokens ---')
    gen(s, continue_final_message=True)

In [ ]:
s=[{'role':'system', 'content': ''},
   {'role':'user', 'content':'The capital of France is Paris. What is the capital of Germany?'},
   {'role':'assistant', 'content':'Sure thing meatbag'+think_suffix}]
(gen(s))

s=[{'role':'system', 'content': ''},
   {'role':'user', 'content':'The capital of France is Paris. What is the capital of Germany?'},
   {'role':'assistant', 'content':'Sure thing meatbag'}]
(gen(s))


s=[{'role':'system', 'content': 'You are the greatest storyteller in the world :) :O'},
   {'role':'user', 'content':'Tell me a long story about where is coconut?'+think_suffix},]
   # {'role':'assistant', 'content':think_suffix+'Sure thing meatbag'}]
(gen(s))

s=[{'role':'system', 'content': 'You are the greatest storyteller in the world :) :O'},
   {'role':'user', 'content':'Tell me a long story about where is coconut?'},]
   # {'role':'assistant', 'content':think_suffix+'Sure thing meatbag'}]
(gen(s))

In [ ]:
(gen('Explain the theory of relativity in simple terms.\nA:<|start-latent|><|latent|><|end-latent|>'))
(gen('Explain the theory of relativity in simple terms.\nA:'));